In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

path_folder = '/content/drive/MyDrive/'

files = [
    'hasil_preprocessing_lowercase.csv'
]

In [ ]:
df = pd.read_csv(path_folder + files[0])
df

#Membaca Sentimen dari Isi Berita (data_clean)

In [ ]:
import pandas as pd
from transformers import pipeline
import os

# 1. Load Model
model_checkpoint = "mmhhz/model-indobert"
classifier = pipeline("text-classification", model=model_checkpoint)

# 2. Ambil Label dan Confidence Score
def get_sentiment_details(teks):
    if pd.isna(teks) or teks == '':
        return "NEUTRAL", 0.0

    # Ambil hasil prediksi
    hasil = classifier(str(teks)[:512])
    label = hasil[0]['label']
    score_pc = round(hasil[0]['score'] * 100, 2)

    return label, score_pc

In [ ]:
# 3. Eksekusi ke Dataframe
print("Sedang melabeli data berita... ")


df[['sentimen', 'nilai_akurasi']] = df['data_clean'].apply(
    lambda x: pd.Series(get_sentiment_details(x))
)

print("Selesai memberikan label!")

In [ ]:
df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


plt.figure(figsize=(8,5))

ax = sns.countplot(x='sentimen', data=df, palette='viridis')


for container in ax.containers:
    ax.bar_label(container, padding=3)

plt.title('Distribusi Sentimen Berita Politik (IndoBERT Fine-Tuning)')
plt.ylabel('Jumlah Berita')
plt.xlabel('Kategori Sentimen')


plt.ylim(0, df['sentimen'].value_counts().max() * 1.1)

plt.show()

In [ ]:
# 4. Cek hasil
print("\nHasil Labeling:")
print(df['sentimen'].value_counts())

In [ ]:
# Simpan hasil gabungan kembali ke Drive agar tidak hilang saat session Colab habis
df.to_csv(path_folder + "hasil_labeling_IndoBERT.csv", index=False)